# Data Cleaning
This ipynb file is to do the data cleaning for the file (world-education-dataset.csv)

### 1. Load the Dataset

In [1]:
import pandas as pd
import numpy as np

# 1. Load the dataset
file_path = "world-education-data.csv"
df = pd.read_csv(file_path)

# Display the first 5 rows to confirm it loaded correctly
print("--- FIRST 5 ROWS OF THE DATASET ---")
display(df.head())

--- FIRST 5 ROWS OF THE DATASET ---


,country,country_code,year,gov_exp_pct_gdp,lit_rate_adult_pct,pri_comp_rate_pct,pupil_teacher_primary,pupil_teacher_secondary,school_enrol_primary_pct,school_enrol_secondary_pct,school_enrol_tertiary_pct
0,Afghanistan,AFG,1999,NaN,NaN,NaN,33.18571,NaN,27.298849,NaN,NaN
1,Afghanistan,AFG,2000,NaN,NaN,NaN,NaN,NaN,22.162991,NaN,NaN
2,Afghanistan,AFG,2001,NaN,NaN,NaN,NaN,NaN,22.908590,14.47151,NaN
3,Afghanistan,AFG,2002,NaN,NaN,NaN,NaN,NaN,75.959747,NaN,NaN
4,Afghanistan,AFG,2003,NaN,NaN,NaN,NaN,NaN,96.553680,14.07805,1.38107


### 2. Check for Missing Values

In [7]:
# 2. Dataset Checking
print("--- DATASET SHAPE (Rows, Columns) ---")
print(df.shape)  # Should verify it has over 3,000 rows and 11 columns

print("\n--- DATA TYPES AND COMPLIANCE CHECK ---")
print(df.info())

print("\n--- MISSING VALUES PER ATTRIBUTE ---")
missing_values = df.isnull().sum()
print(missing_values)

print("\n--- SUMMARY STATISTICS FOR NUMERICAL COLUMNS ---")
display(df.describe())

--- DATASET SHAPE (Rows, Columns) ---
(5892, 11)

--- DATA TYPES AND COMPLIANCE CHECK ---
<class 'pandas.DataFrame'>
RangeIndex: 5892 entries, 0 to 5891
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   country                     5892 non-null   str    
 1   country_code                5892 non-null   str    
 2   year                        5892 non-null   int64  
 3   gov_exp_pct_gdp             4499 non-null   float64
 4   lit_rate_adult_pct          1877 non-null   float64
 5   pri_comp_rate_pct           4440 non-null   float64
 6   pupil_teacher_primary       3676 non-null   float64
 7   pupil_teacher_secondary     3017 non-null   float64
 8   school_enrol_primary_pct    5352 non-null   float64
 9   school_enrol_secondary_pct  4745 non-null   float64
 10  school_enrol_tertiary_pct   4392 non-null   float64
dtypes: float64(8), int64(1), str(2)
memory usage: 506.5 KB
None

--- MIS

,year,gov_exp_pct_gdp,lit_rate_adult_pct,pri_comp_rate_pct,pupil_teacher_primary,pupil_teacher_secondary,school_enrol_primary_pct,school_enrol_secondary_pct,school_enrol_tertiary_pct
count,5892.000000,4499.000000,1877.000000,4440.000000,3676.000000,3017.000000,5352.000000,4745.000000,4392.000000
mean,2010.921419,4.320129,79.483333,87.776740,25.344398,17.560340,101.525234,78.939810,36.533796
std,7.119808,1.736997,17.186877,17.857748,12.780357,7.465528,13.029901,28.350998,26.960123
min,1999.000000,0.242600,14.000000,14.411250,5.360520,4.979320,8.447979,3.293810,0.117370
25%,2005.000000,3.180390,65.984039,80.308426,15.888230,11.983680,97.281084,59.364799,12.605780
50%,2011.000000,4.101967,83.915154,94.604504,22.172125,16.224470,101.556335,85.707581,30.962285
75%,2017.000000,5.163850,94.215561,99.081194,32.569205,21.583330,106.822365,99.230003,57.325488
max,2023.000000,15.863470,100.000000,156.167175,100.236490,80.052320,257.434204,194.460022,166.665649


### 3. Dataset Cleaning

In [10]:
# 3. Improved Dataset Cleaning

# Create a copy of the dataframe to keep the raw data safe
df_clean = df.copy()

# Standardise column names just in case
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

# Required columns for the dashboard
required_columns = [
    "country",
    "country_code",
    "year",
    "gov_exp_pct_gdp",
    "lit_rate_adult_pct",
    "pri_comp_rate_pct",
    "pupil_teacher_primary",
    "pupil_teacher_secondary",
    "school_enrol_primary_pct",
    "school_enrol_secondary_pct",
    "school_enrol_tertiary_pct"
]

# Check if required columns exist
missing_required = [col for col in required_columns if col not in df_clean.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df_clean = df_clean[required_columns].copy()

# Clean text columns
df_clean["country"] = df_clean["country"].astype(str).str.strip()
df_clean["country_code"] = df_clean["country_code"].astype(str).str.strip().str.upper()

# Convert year
df_clean["year"] = pd.to_numeric(df_clean["year"], errors="coerce").astype("Int64")

# Numeric columns
numerical_cols = [
    "gov_exp_pct_gdp",
    "lit_rate_adult_pct",
    "pri_comp_rate_pct",
    "pupil_teacher_primary",
    "pupil_teacher_secondary",
    "school_enrol_primary_pct",
    "school_enrol_secondary_pct",
    "school_enrol_tertiary_pct"
]

for col in numerical_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Drop rows with missing important identifiers
df_clean = df_clean.dropna(subset=["country", "country_code", "year"]).copy()

# Remove duplicate country-year rows
duplicate_count = df_clean.duplicated(subset=["country_code", "year"]).sum()
print("Duplicate country-year rows before removal:", duplicate_count)

df_clean = df_clean.drop_duplicates(subset=["country_code", "year"], keep="first").copy()

# Remove regional / income-group aggregate rows
# These are not individual countries and can distort country comparisons and maps
aggregate_codes = set("""
AFE AFW ARB CSS CEB EAR EAS EAP TEA EMU ECS ECA TEC EUU FCS HPC HIC IBD IBT
IDB IDX IDA LTE LCN LAC TLA LDC LMY LIC LMC MEA MNA TMN MIC NAC INX OED OSS
PSS PST PRE SST SAS TSA SSF SSA TSS UMC WLD
""".split())

aggregates_removed = df_clean[df_clean["country_code"].isin(aggregate_codes)].copy()
df_clean = df_clean[~df_clean["country_code"].isin(aggregate_codes)].copy()

print("Aggregate / regional rows removed:", len(aggregates_removed))
print("Rows after removing aggregates:", len(df_clean))
print("Unique countries remaining:", df_clean["country_code"].nunique())

# Check missing values before imputation
missing_before = df_clean[numerical_cols].isna().sum().reset_index()
missing_before.columns = ["column", "missing_before_imputation"]

display(missing_before)

Duplicate country-year rows before removal: 0
Aggregate / regional rows removed: 1198
Rows after removing aggregates: 4694
Unique countries remaining: 208


,column,missing_before_imputation
0,gov_exp_pct_gdp,1179
1,lit_rate_adult_pct,3825
2,pri_comp_rate_pct,1434
3,pupil_teacher_primary,1938
4,pupil_teacher_secondary,2592
5,school_enrol_primary_pct,535
6,school_enrol_secondary_pct,1137
7,school_enrol_tertiary_pct,1472


In [12]:
# 4. Validate numeric ranges

# Note:
# Enrollment and completion rates are allowed to exceed 100 because they are likely gross rates.
# So we should not force them to 100 maximum.

valid_ranges = {
    "gov_exp_pct_gdp": (0, 50),
    "lit_rate_adult_pct": (0, 100),
    "pri_comp_rate_pct": (0, 200),
    "pupil_teacher_primary": (0, 150),
    "pupil_teacher_secondary": (0, 150),
    "school_enrol_primary_pct": (0, 300),
    "school_enrol_secondary_pct": (0, 300),
    "school_enrol_tertiary_pct": (0, 300)
}

invalid_summary = []

for col, (low, high) in valid_ranges.items():
    invalid_mask = df_clean[col].notna() & (
        (df_clean[col] < low) | (df_clean[col] > high)
    )
    
    invalid_count = invalid_mask.sum()
    
    invalid_summary.append({
        "column": col,
        "invalid_values_set_to_missing": int(invalid_count)
    })
    
    # Set invalid values to NaN so they can be handled by imputation
    df_clean.loc[invalid_mask, col] = np.nan

invalid_report = pd.DataFrame(invalid_summary)

display(invalid_report)

,column,invalid_values_set_to_missing
0,gov_exp_pct_gdp,0
1,lit_rate_adult_pct,0
2,pri_comp_rate_pct,0
3,pupil_teacher_primary,0
4,pupil_teacher_secondary,0
5,school_enrol_primary_pct,0
6,school_enrol_secondary_pct,0
7,school_enrol_tertiary_pct,0


In [14]:
# 5. Create imputation flags before filling missing values

for col in numerical_cols:
    df_clean[f"{col}_was_imputed"] = df_clean[col].isna()

df_clean.head()

,country,country_code,year,gov_exp_pct_gdp,lit_rate_adult_pct,pri_comp_rate_pct,pupil_teacher_primary,pupil_teacher_secondary,school_enrol_primary_pct,school_enrol_secondary_pct,school_enrol_tertiary_pct,gov_exp_pct_gdp_was_imputed,lit_rate_adult_pct_was_imputed,pri_comp_rate_pct_was_imputed,pupil_teacher_primary_was_imputed,pupil_teacher_secondary_was_imputed,school_enrol_primary_pct_was_imputed,school_enrol_secondary_pct_was_imputed,school_enrol_tertiary_pct_was_imputed
0,Afghanistan,AFG,1999,NaN,NaN,NaN,33.18571,NaN,27.298849,NaN,NaN,True,True,True,False,True,False,True,True
1,Afghanistan,AFG,2000,NaN,NaN,NaN,NaN,NaN,22.162991,NaN,NaN,True,True,True,True,True,False,True,True
2,Afghanistan,AFG,2001,NaN,NaN,NaN,NaN,NaN,22.908590,14.47151,NaN,True,True,True,True,True,False,False,True
3,Afghanistan,AFG,2002,NaN,NaN,NaN,NaN,NaN,75.959747,NaN,NaN,True,True,True,True,True,False,True,True
4,Afghanistan,AFG,2003,NaN,NaN,NaN,NaN,NaN,96.553680,14.07805,1.38107,True,True,True,True,True,False,False,False


In [16]:
# 6. Safer missing value imputation

df_clean = df_clean.sort_values(["country_code", "year"]).reset_index(drop=True)

# Step 1:
# Interpolate missing values only between known values within the same country.
# This is safer than blindly copying values across many years.
df_clean[numerical_cols] = (
    df_clean.groupby("country_code", group_keys=False)[numerical_cols]
    .apply(lambda group: group.interpolate(method="linear", limit_area="inside"))
)

# Step 2:
# Fill only short gaps using nearby years within the same country.
# limit=2 means it only fills up to 2 years forward/backward, not 10+ years.
df_clean[numerical_cols] = (
    df_clean.groupby("country_code", group_keys=False)[numerical_cols]
    .apply(lambda group: group.ffill(limit=2).bfill(limit=2))
)

# Step 3:
# Fill remaining missing values using the median of the same year across countries.
# This is better than one global median because education indicators change over time.
for col in numerical_cols:
    yearly_median = df_clean.groupby("year")[col].transform("median")
    df_clean[col] = df_clean[col].fillna(yearly_median)

# Step 4:
# Final fallback using overall median only if anything is still missing.
for col in numerical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Round for cleaner dashboard display
df_clean[numerical_cols] = df_clean[numerical_cols].round(2)

print("--- Missing values after imputation ---")
print(df_clean[numerical_cols].isna().sum())

--- Missing values after imputation ---
gov_exp_pct_gdp               0
lit_rate_adult_pct            0
pri_comp_rate_pct             0
pupil_teacher_primary         0
pupil_teacher_secondary       0
school_enrol_primary_pct      0
school_enrol_secondary_pct    0
school_enrol_tertiary_pct     0
dtype: int64


In [18]:
# 7. Create cleaning report

missing_after = df_clean[numerical_cols].isna().sum().reset_index()
missing_after.columns = ["column", "missing_after_imputation"]

cleaning_report = missing_before.merge(missing_after, on="column", how="left")
cleaning_report = cleaning_report.merge(invalid_report, on="column", how="left")

cleaning_report["original_rows"] = len(df)
cleaning_report["cleaned_rows"] = len(df_clean)
cleaning_report["aggregate_rows_removed"] = len(aggregates_removed)
cleaning_report["unique_countries"] = df_clean["country_code"].nunique()
cleaning_report["year_min"] = int(df_clean["year"].min())
cleaning_report["year_max"] = int(df_clean["year"].max())

display(cleaning_report)

,column,missing_before_imputation,missing_after_imputation,invalid_values_set_to_missing,original_rows,cleaned_rows,aggregate_rows_removed,unique_countries,year_min,year_max
0,gov_exp_pct_gdp,1179,0,0,5892,4694,1198,208,1999,2023
1,lit_rate_adult_pct,3825,0,0,5892,4694,1198,208,1999,2023
2,pri_comp_rate_pct,1434,0,0,5892,4694,1198,208,1999,2023
3,pupil_teacher_primary,1938,0,0,5892,4694,1198,208,1999,2023
4,pupil_teacher_secondary,2592,0,0,5892,4694,1198,208,1999,2023
5,school_enrol_primary_pct,535,0,0,5892,4694,1198,208,1999,2023
6,school_enrol_secondary_pct,1137,0,0,5892,4694,1198,208,1999,2023
7,school_enrol_tertiary_pct,1472,0,0,5892,4694,1198,208,1999,2023


In [20]:
# 8. Final quality checks

print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print("Unique countries:", df_clean["country_code"].nunique())
print("Year range:", df_clean["year"].min(), "-", df_clean["year"].max())

print("\nMissing values in dashboard columns:")
print(df_clean[required_columns].isna().sum())

print("\nDuplicate country-year rows:")
print(df_clean.duplicated(subset=["country_code", "year"]).sum())

assert df_clean["country"].isna().sum() == 0
assert df_clean["country_code"].isna().sum() == 0
assert df_clean["year"].isna().sum() == 0
assert df_clean.duplicated(subset=["country_code", "year"]).sum() == 0
assert df_clean[numerical_cols].isna().sum().sum() == 0
assert len(df_clean) >= 3000
assert len(required_columns) >= 10

print("\nAll quality checks passed.")

Rows: 4694
Columns: 19
Unique countries: 208
Year range: 1999 - 2023

Missing values in dashboard columns:
country                       0
country_code                  0
year                          0
gov_exp_pct_gdp               0
lit_rate_adult_pct            0
pri_comp_rate_pct             0
pupil_teacher_primary         0
pupil_teacher_secondary       0
school_enrol_primary_pct      0
school_enrol_secondary_pct    0
school_enrol_tertiary_pct     0
dtype: int64

Duplicate country-year rows:
0

All quality checks passed.


In [22]:
# Check if aggregate / regional rows still exist

aggregate_codes = set("""
AFE AFW ARB CSS CEB EAR EAS EAP TEA EMU ECS ECA TEC EUU FCS HPC HIC IBD IBT
IDB IDX IDA LTE LCN LAC TLA LDC LMY LIC LMC MEA MNA TMN MIC NAC INX OED OSS
PSS PST PRE SST SAS TSA SSF SSA TSS UMC WLD
""".split())

remaining_aggregates = df_clean[df_clean["country_code"].isin(aggregate_codes)]

print("Remaining aggregate/regional rows:", len(remaining_aggregates))

remaining_aggregates[["country", "country_code", "year"]].head(20)

Remaining aggregate/regional rows: 0


,country,country_code,year


In [28]:
# 9. Save final cleaned files

# Dashboard version: only the columns needed for D3.js
dashboard_df = df_clean[required_columns].copy()

dashboard_df.to_csv("cleaned_world_education_dataset.csv", index=False)

# Full version: includes imputation flags for documentation/explanation
df_clean.to_csv("world_education_clean_with_flags.csv", index=False)

# Cleaning report
cleaning_report.to_csv("cleaning_report.csv", index=False)

print("Saved dashboard-ready file: cleaned_world_education_dataset.csv")
print("Saved full cleaned file with flags: world_education_clean_with_flags.csv")
print("Saved cleaning report: cleaning_report.csv")

Saved dashboard-ready file: cleaned_world_education_dataset.csv
Saved full cleaned file with flags: world_education_clean_with_flags.csv
Saved cleaning report: cleaning_report.csv
